In [1]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.5.1
cuda available: False


In [2]:
from datasets import load_dataset

data_files = {
    "train": "../data/simulated_s1_gap_between/train.jsonl",
    "validation": "../data/simulated_s1_gap_between/valid.jsonl",
}

ds = load_dataset("json", data_files=data_files)
ds


/opt/modules/i12g/anaconda/envs/sysgen_p06/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['case', 'a_start', 'b_start', 'spacing', 'gap_count', 'gap_region', 'sequence'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['case', 'a_start', 'b_start', 'spacing', 'gap_count', 'gap_region', 'sequence'],
        num_rows: 2000
    })
})

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

# vocab: base + gap + special tokens
vocab = {
    "[PAD]": 0,
    "[UNK]": 1,
    "[CLS]": 2,
    "[SEP]": 3,
    "[MASK]": 4,
    "A": 5, "C": 6, "G": 7, "T": 8, "-": 9,
}

tok = Tokenizer(WordLevel(vocab=vocab, unk_token="[UNK]"))
tok.pre_tokenizer = Whitespace()

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)

def to_spaced(seq: str) -> str:
    return " ".join(list(seq))

# quick test
print(tokenizer(to_spaced("ACGT--TA")))


{'input_ids': [5, 6, 7, 8, 9, 9, 8, 5], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [4]:
def tokenize_batch(batch):
    x = [to_spaced(s) for s in batch["sequence"]]
    return tokenizer(x, truncation=True)

train_small = ds["train"].shuffle(seed=0).select(range(5000))
valid_small = ds["validation"].shuffle(seed=0).select(range(500))

train_tok = train_small.map(tokenize_batch, batched=True, remove_columns=train_small.column_names)
valid_tok = valid_small.map(tokenize_batch, batched=True, remove_columns=valid_small.column_names)

train_tok, valid_tok


(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 5000
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 500
 }))

In [5]:
from transformers import BertConfig, BertForMaskedLM

config = BertConfig(
    vocab_size=len(vocab),
    hidden_size=128,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=256,
    max_position_embeddings=512,
    pad_token_id=vocab["[PAD]"],
)

model = BertForMaskedLM(config)
print("params:", sum(p.numel() for p in model.parameters())/1e6, "M")


params: 0.614026 M


In [6]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

args = TrainingArguments(
    output_dir="../experiments/mlm_s1_quick",
    per_device_train_batch_size=64 if torch.cuda.is_available() else 16,
    per_device_eval_batch_size=64 if torch.cuda.is_available() else 16,
    num_train_epochs=2,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    data_collator=data_collator,
)

trainer.train()


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss
100,1.376200,1.376622
200,1.363800,1.368361
300,1.357600,1.358919
400,1.352100,1.359494
500,1.347700,1.355472
600,1.348000,1.349359


TrainOutput(global_step=626, training_loss=1.3682734455925207, metrics={'train_runtime': 81.0205, 'train_samples_per_second': 123.425, 'train_steps_per_second': 7.726, 'total_flos': 6563448000000.0, 'train_loss': 1.3682734455925207, 'epoch': 2.0})

In [18]:
import numpy as np, random, json

cfg = json.load(open("../data/simulated_s1_gap_between/config.json"))
motif_a = cfg["motif_a"]
motif_b = cfg["motif_b"]
la, lb = len(motif_a), len(motif_b)

def overwrite_str(s, start, sub):
    s = list(s)
    for i,ch in enumerate(sub):
        s[start+i] = ch
    return "".join(s)

def make_version(seq_bg, b_pos):
    return overwrite_str(seq_bg, b_pos, motif_b)

def score_B_at(seq_with_B, b_pos):
    return avg_logprob_masked_motif(seq_with_B, motif_b, b_pos)

def fair_delta(example, invalid_shift=1):
    seq_bg = example["sequence"]
    a = example["a_start"]
    b = example["b_start"]
    if b < 0:
        return None
    b2 = b + invalid_shift
    if b2 + lb > len(seq_bg):
        return None

    a_end = a + la
    if b2 < a_end:
        return None

    seq_valid = make_version(seq_bg, b)
    seq_inv   = make_version(seq_bg, b2)

    s_valid = score_B_at(seq_valid, b)
    s_inv   = score_B_at(seq_inv, b2)

    return s_valid - s_inv, s_valid, s_inv, b, b2

both_ids = [i for i,r in enumerate(ds["validation"]) if r["case"]=="both_valid"]
ex = ds["validation"][random.choice(both_ids)]
print(fair_delta(ex, invalid_shift=1))


(-0.3224724595035826, -1.6204547541482108, -1.2979822946446282, 107, 108)
